# MGSM replication: Kaggle runner

Runs `src/run.py` from the `rio_project` repo on a free Kaggle GPU. Pass 1 for one model:
11 languages x 250 items x 3 cells (DIRECT, NATIVE-COT, EN-COT) = 8,250 generations.

**Notebook settings (right-hand panel):** Accelerator = GPU T4 x2 (x1 is fine for the 0.6B and 1.7B
models), Internet = ON. For the full run use **Save Version -> Save & Run All (Commit)** so it runs
headless (max 12 h); results then appear under the notebook's *Output* tab.

Rows are appended to the JSONL as each (language, cell) finishes and the run resumes if restarted.

In [ ]:
# 1. Get the code from GitHub and move into the project folder.
REPO_URL = "https://github.com/tomf0/rio_project.git"
PROJECT = "paper_replication/mgsm_paper/tom"
CONFIG = "config/kaggle_qwen3_8b_pass1.yaml"   # 8B needs Accelerator = GPU T4 x2 (tensor_parallel: 2)   # <- the one line to change per model

import os, subprocess
os.chdir("/kaggle/working")
if not os.path.isdir("repo"):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "repo"], check=True)
else:
    subprocess.run(["git", "-C", "repo", "pull"], check=True)
os.chdir(f"/kaggle/working/repo/{PROJECT}")
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
!ls

In [ ]:
# 2. Dependencies. vLLM is large (several minutes) and brings its own torch build.
#    Kaggle's preinstalled torchaudio was built against a different CUDA and makes transformers
#    crash on import; we never use audio, so remove it. If vLLM fails, the run falls back to hf.
!pip install -q pyyaml
!pip install -q vllm
!pip uninstall -y -q torchaudio

import importlib
BACKEND = "vllm" if importlib.util.find_spec("vllm") else "hf"
print("backend:", BACKEND)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 3. Quick check: 3 items x 2 languages x 3 cells on the real GPU. Read the completions.
!python -m src.run {CONFIG} backend={BACKEND} n_items=3 languages=[de,sw] output=results/kaggle/check.jsonl
import json
for r in map(json.loads, open("results/kaggle/check.jsonl", encoding="utf-8")):
    print(f"{r['lang']} {r['cell']:11} item {r['item_id']} gold {r['gold']:6} got {r['extracted_answer']!s:8} "
          f"{r['extraction_method']!s:22} {r['finish_reason']:6} | {r['completion'][:90]!r}")

In [ ]:
# 4. Main run. Resumable: rerunning skips rows already in the file.
EXTRA = "" if BACKEND == "vllm" else "batch_size=16"
!python -m src.run {CONFIG} backend={BACKEND} {EXTRA}

In [ ]:
# 5. Summary table (accuracy per language x cell, point estimates) and zip for download.
import json, collections
import yaml
cfg_out = yaml.safe_load(open(CONFIG))["output"]
rows = [json.loads(l) for l in open(cfg_out, encoding="utf-8")]
acc = collections.defaultdict(list)
for r in rows:
    acc[(r["cell"], r["lang"])].append(bool(r["correct"]))
langs = ["en", "de", "fr", "es", "ru", "zh", "ja", "th", "te", "bn", "sw"]
print(f"{'cell':11}" + "".join(f"{l:>6}" for l in langs) + "   avg")
for cell in ["direct", "native_cot", "en_cot"]:
    vals = [100 * sum(acc[(cell, l)]) / len(acc[(cell, l)]) if acc[(cell, l)] else float('nan') for l in langs]
    print(f"{cell:11}" + "".join(f"{v:6.1f}" for v in vals) + f"  {sum(vals)/len(vals):5.1f}")
print(f"\n{len(rows)} rows; finish reasons:", collections.Counter(r["finish_reason"] for r in rows))
print("extraction methods:", collections.Counter(str(r["extraction_method"]) for r in rows))
!cd /kaggle/working && zip -q -r results.zip repo/paper_replication/mgsm_paper/tom/results/kaggle && ls -lh results.zip